# НИС «Основы анализа данных в Python»

*Алла Тамбовцева*

## Практикум 14*. Коэффициенты корреляции и значимость

Импортируем библиотеку `pandas` и модуль `stats` из библиотеки `scipy`:

In [1]:
import pandas as pd
from scipy import stats

Загрузим данные из основного практикума 14:

In [2]:
df = pd.read_csv("Griliches.csv")
df.head()

,rownames,rns,rns80,mrt,mrt80,smsa,smsa80,med,iq,kww,...,age,age80,school,school80,expr,expr80,tenure,tenure80,lw,lw80
0,1,no,no,no,yes,yes,yes,8,93,35,...,19,31,12,12,0.462,10.635,0,2,5.900,6.645
1,2,no,no,no,yes,yes,yes,14,119,41,...,23,37,16,18,0.000,11.367,2,16,5.438,6.694
2,3,no,no,no,yes,yes,yes,14,108,46,...,20,33,14,14,0.423,11.035,1,9,5.710,6.715
3,4,no,no,no,yes,yes,yes,12,96,32,...,18,32,12,12,0.333,13.089,1,7,5.481,6.477
4,5,no,no,yes,yes,yes,yes,6,74,27,...,26,34,9,11,9.013,14.402,3,5,5.927,6.332


Вспомним, как вычислить коэффициенты корреляции Пирсона между возрастом и заработной платой по группам (отдельно по каждому году):

In [3]:
df.groupby("year")[["age", "lw"]].apply(lambda x: x.corr().iloc[0, 1].round(2))

year
66    0.30
67    0.53
68    0.44
69    0.65
70    0.56
71    0.47
73    0.38
dtype: float64

Как помимо коэффициентов получить значения p-value, позволяющие дать ответ на вопрос о наличии статистически значимой линейной связи? Для начала воспользуемся функцией `pearsonr()` из модуля `stats` и применим ее к показателям без деления на годы:

In [4]:
stats.pearsonr(df["age"], df["lw"])

(0.5345050220117539, 3.169972914765287e-57)

Коэффициент Пирсона равен 0.53, p-value примерно 0, нулевая гипотеза об отсутствии линейной связи отвергается.

Как можно заметить, функция `pearsonr()` применяется к двум отдельным столбцам или массивам. Если мы захотим применить ее к результату группировки, логичнее будет столбцы выбирать не перед вычислением корреляции, а «во время»:

In [5]:
# сначала группируем,
# потом внутри lambda-функции выбираем столбцы,
# x здесь – датафрейм с кучей столбцов за каждый год

df.groupby("year").apply(lambda x: stats.pearsonr(x["age"], x["lw"]))

year
66     (0.30157462818678393, 6.10985763851867e-06)
67      (0.5265885449016506, 9.28706352568352e-06)
68     (0.44266328717428993, 4.40915925534968e-05)
69    (0.6458206600521558, 2.4931229615230825e-11)
70    (0.5564410743137235, 1.8036399896971194e-06)
71     (0.4666839461919918, 2.744506997103302e-06)
73      (0.3822058627182967, 7.21206087559523e-07)
dtype: object

Результат понятный, но не совсем изящный: для каждого года был вычислен коэффициент Пирсона и p-value без округления. Причем, так как функция `pearsonr()` возвращает кортеж, полученный результат – это текстовый столбец (списки и кортежи в ячейках последовательностей и датафреймов `pandas` хранит в текстовом виде, «сливая» все элементы в одну большу строку с запятыми и скобками).

Давайте перепишем код выше, чтобы с его помощью можно было:

* видеть результаты с точностью до сотых;
* вычислять коэффициенты корреляции между любыми парами столбцов, не только `age` и `lw`.

Напишем свою функцию, которая будет принимать на вход датафрейм `x`, выбирать оттуда первый и второй столбец по отдельности (считаем, что в `x` мы сохраняем два предварительно выбранных столбца) и подставлять их в `pearsonr()`:

In [6]:
# выбираем столбец с индексом 0, 
# выбираем столбец с индексом 1,
# подставляем в pearsonr(),
# сохраняем результаты по отдельности и округляем

def get_corr_pvalue(x):
    r, pvalue = stats.pearsonr(x.iloc[:, 0], x.iloc[:, 1])
    return round(r, 2), round(pvalue, 2)

Применим функцию к результату группировки, предварительно выбрав два столбца, `age` и `lw`:

In [7]:
df.groupby("year")[["age", "lw"]].apply(get_corr_pvalue)

year
66     (0.3, 0.0)
67    (0.53, 0.0)
68    (0.44, 0.0)
69    (0.65, 0.0)
70    (0.56, 0.0)
71    (0.47, 0.0)
73    (0.38, 0.0)
dtype: object

Выберем другие два столбца – `iq` и `lw`, там p-value поинтереснее, не все нулевые:

In [8]:
df.groupby("year")[["iq", "lw"]].apply(get_corr_pvalue)

year
66    (0.17, 0.01)
67     (0.39, 0.0)
68    (0.14, 0.21)
69     (0.36, 0.0)
70    (0.29, 0.02)
71    (0.25, 0.02)
73     (0.24, 0.0)
dtype: object

Выглядит поприятнее, но это все еще текстовый столбец. Чтобы разбить результаты на два столбца, нужно превратить кортеж, возвращаемый `pearsonr()`, в последовательность `pandas Series` из двух значений. Условно, это будет одна строка таблицы с двумя элементами, первое – коэффициент корреляции, второе – p-value. Учтем это внутри функции:

In [9]:
# кортеж – результат pearsonr()
# помещаем в Series()

def get_corr_pvalue(x):
    dat = pd.Series(stats.pearsonr(x.iloc[:, 0], x.iloc[:, 1]))
    return dat.round(2)

Применим обновленную функцию:

In [10]:
df.groupby("year")[["iq", "lw"]].apply(get_corr_pvalue)

,0,1
year,,
66,0.17,0.01
67,0.39,0.00
68,0.14,0.21
69,0.36,0.00
70,0.29,0.02
71,0.25,0.02
73,0.24,0.00


Уже красиво. Можем добавить названия столбцов вместо 0 и 1:

In [11]:
res = df.groupby("year")[["iq", "lw"]].apply(get_corr_pvalue)
res.columns = ["coef", "p-value"]
res

,coef,p-value
year,,
66,0.17,0.01
67,0.39,0.00
68,0.14,0.21
69,0.36,0.00
70,0.29,0.02
71,0.25,0.02
73,0.24,0.00


Однако иногда сами p-value не указывают, а вместо них ставят «звездочки», обозначающие значимость коэффициентов корреляции при разных уровнях значимости $\alpha$. Чаще всего используют такие соответствия:

* значимость на 1%-ном уровне значимости – `***`;
* значимость на 5%-ном уровне значимости – `**`;
* значимость на 10%-ном уровне значимости – `*`.

Напишем функцию, которая вычисляет коэффициент корреляции Пирсона и соответствующее p-value, а затем, сравнивая p-value с разными уровнями значимости, доклеивает к коэффициенту необходимое число «звездочек»:

In [12]:
# округляем коэффициент, превращаем в строку и добавляем звездочки
# на выходе снова – pandas Series, но теперь без p-value

def get_corr_stars(x):
    r, pvalue = stats.pearsonr(x.iloc[:, 0], x.iloc[:, 1])
    
    if pvalue < 0.01:
        star = " ***"
    elif pvalue < 0.05:
        star = " **"
    elif pvalue < 0.1:
        star = " *"
    else:
        star = ""
    
    coef = str(round(r, 2)) + star
    dat = pd.Series([coef])
    return dat

Применим функцию к результатам группировки:

In [13]:
res_stars = df.groupby("year")[["iq", "lw"]].apply(get_corr_stars)
res_stars.columns = ["coef"]
res_stars

,coef
year,
66,0.17 **
67,0.39 ***
68,0.14
69,0.36 ***
70,0.29 **
71,0.25 **
73,0.24 ***


Из-за того, что число «звездочек» не везде одинаковое, значения в столбце не выглядят выровненными. Однако, если мы будем выгружать эту табличку в LaTeX или Markdown, все будет нормально:

In [14]:
# тут пробелы ни на что не влияют
print(res_stars.to_latex())

\begin{tabular}{ll}
\toprule
{} &      coef \\
year &           \\
\midrule
66   &   0.17 ** \\
67   &  0.39 *** \\
68   &      0.14 \\
69   &  0.36 *** \\
70   &   0.29 ** \\
71   &   0.25 ** \\
73   &  0.24 *** \\
\bottomrule
\end{tabular}



In [15]:
# тут само выравнивается
print(res_stars.to_markdown())

|   year | coef     |
|-------:|:---------|
|     66 | 0.17 **  |
|     67 | 0.39 *** |
|     68 | 0.14     |
|     69 | 0.36 *** |
|     70 | 0.29 **  |
|     71 | 0.25 **  |
|     73 | 0.24 *** |


Если скопируем код Markdown в ячейку Jupyter, получим таблицу с корректным выравниванием:

|   year | coef     |
|-------:|:---------|
|     66 | 0.17 **  |
|     67 | 0.39 *** |
|     68 | 0.14     |
|     69 | 0.36 *** |
|     70 | 0.29 **  |
|     71 | 0.25 **  |
|     73 | 0.24 *** |

Поскольку язык HTML тоже не чувствителен к пробелам, и здесь проблем быть не должно:

In [16]:
print(res_stars.to_html())

<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>coef</th>
    </tr>
    <tr>
      <th>year</th>
      <th></th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>66</th>
      <td>0.17 **</td>
    </tr>
    <tr>
      <th>67</th>
      <td>0.39 ***</td>
    </tr>
    <tr>
      <th>68</th>
      <td>0.14</td>
    </tr>
    <tr>
      <th>69</th>
      <td>0.36 ***</td>
    </tr>
    <tr>
      <th>70</th>
      <td>0.29 **</td>
    </tr>
    <tr>
      <th>71</th>
      <td>0.25 **</td>
    </tr>
    <tr>
      <th>73</th>
      <td>0.24 ***</td>
    </tr>
  </tbody>
</table>


И да, код HTML можно выгрузить в файл с расширением `.htm`, который открывается в Word и подобных редакторах:

In [17]:
# ищем в папке файл corr_tab.htm, 
# по умолчанию откроется в браузере,
# но можно открыть с помощью Word и редактировать

res_stars.to_html("corr_tab.htm")